# WAT.ai — Adaptive TTS: Inference & KV-Cache Analysis
**Research notebook · Rababb · dots.tts · NVIDIA T4 / FP16**

**Goal:** Trace the inference path, locate repeated DiT evaluations, map tensor dependencies and reuse, inspect the existing KV-cache mechanism, and benchmark eager versus optimized inference as a baseline for adaptive NFE.

**Pinned configuration**

| Parameter | Value |
|---|---|
| Source commit | `3cb6f94a571097b500038f1619959075282c5c98` |
| Model revision | `b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1` |
| GPU / precision | Colab NVIDIA T4 / FP16 |
| Solver | Euler, 10 NFE, guidance 1.2, speaker scale 1.5 |
| Prompt | “Hello, this is a test of the dots TTS model.” |
| Random seed | 42 |

**Read first:** The results section includes measurements *recorded in the original notebook*. Running this cleaned notebook can yield different timings. `optimize=True` turns on optimizations beyond KV caching; the speedup is **not** an isolated KV-cache measurement.


## 1 · Environment & model setup
Run this section first in **Google Colab → GPU (T4)** with **Python 3.12** -> Change runtime to 2026.07. Install dependencies, download the pinned model, and initialize the runtime. The memory-conscious loader is retained from the original notebook because it allowed the checkpoint to load on T4.


In [1]:
import sys

print(sys.version)
assert sys.version_info[:2] == (3, 12), "Use Python 3.12 for this notebook."

!free -h
!nvidia-smi


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.1Gi       6.5Gi       2.0Mi       4.1Gi        10Gi
Swap:             0B          0B          0B
Tue Oct 20 01:35:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off

In [2]:
from pathlib import Path
import subprocess

REPO = Path('/content/dots.tts')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/studio-dots-ai/dots.tts.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '3cb6f94a571097b500038f1619959075282c5c98'], check=True)
%cd /content/dots.tts


/content/dots.tts


NOTE: After the next cell runs, you will have to restart the runtime, but then continue to run the next cell after the one below and it will work.

In [3]:
%pip install -e . "transformers==4.57.0" "setuptools>=68,<82" "accelerate>=1.12.0" pyarrow pandas psutil


Obtaining file:///content/dots.tts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
R

In [1]:
import sys
import gc
import json
import inspect
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import psutil
import soundfile as sf
from IPython.display import Audio, display
from huggingface_hub import snapshot_download

REPO = Path("/content/dots.tts")
OUT = Path("/content/rababb_kv_map")
OUT.mkdir(exist_ok=True)
SOURCE_REVISION = "3cb6f94a571097b500038f1619959075282c5c98"
assert torch.cuda.is_available(), "Enable a GPU runtime."
import subprocess
revision = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert revision == SOURCE_REVISION, "Run Setup to use the expected source revision."

from dots_tts.runtime import DotsTtsRuntime
assert Path(inspect.getfile(DotsTtsRuntime)).is_relative_to(REPO)
print("GPU:", torch.cuda.get_device_name(0))
print("Available CPU RAM (GiB):", round(psutil.virtual_memory().available / 2**30, 2))
print("Available GPU memory (GiB):", round(torch.cuda.mem_get_info()[0] / 2**30, 2))


GPU: Tesla T4
Available CPU RAM (GiB): 7.13
Available GPU memory (GiB): 14.46


### Optional Hugging Face authentication
For publicly accessible assets, leave `USE_HF_TOKEN = False`. If Hugging Face requires authentication, set it to `True` after adding an `HF_TOKEN` secret in Colab. Do **not** put tokens directly into the notebook.


In [2]:
USE_HF_TOKEN = True  # Set True only if access requires your Colab HF_TOKEN secret.

if USE_HF_TOKEN:
    from google.colab import userdata
    from huggingface_hub import login
    token = userdata.get('HF_TOKEN')
    assert token, 'Add HF_TOKEN to Colab secrets before enabling authentication.'
    login(token=token, add_to_git_credential=False)


In [3]:
MODEL_ID = "dots-studio/dots.tts-base"
MODEL_REVISION = "b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1"
checkpoint = Path(snapshot_download(MODEL_ID, revision=MODEL_REVISION))
print("Checkpoint ready.")


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

llm_config.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

speaker_encoder.safetensors:   0%|          | 0.00/29.2M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocoder.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Checkpoint ready.


### T4-compatible model loading
The original model-loading workaround constructs an empty core and streams checkpoint tensors before restoring the upstream loader hooks. **Keep the complete block intact**; it was necessary for the original Colab environment. If the upstream package changes, this workaround may require revalidation.


In [4]:
from accelerate import init_empty_weights
from safetensors import safe_open
import dots_tts.models.dots_tts.model as model_code

Model = model_code.DotsTtsModel
original_core_constructor = model_code.DotsTtsCore
original_artifact_loader = Model._load_artifact_module
original_loader_descriptor = Model.__dict__["_load_artifact_module"]

def create_empty_core(*args, **kwargs):
    with init_empty_weights(include_buffers=False):
        core = original_core_constructor(*args, **kwargs)
    return core

def load_weights(owner, module, path):
    has_empty_parameters = any(p.is_meta for p in module.parameters())
    if not has_empty_parameters:
        return original_artifact_loader(module, path)

    with safe_open(str(path), framework="pt", device="cpu") as checkpoint_file:
        names = list(checkpoint_file.keys())

    weights = {}
    for name in names:
        with safe_open(str(path), framework="pt", device="cpu") as checkpoint_file:
            tensor = checkpoint_file.get_tensor(name)
            dtype = torch.float16 if tensor.is_floating_point() else tensor.dtype
            weights[name] = tensor.to(device="cuda", dtype=dtype)
            del tensor

    # Restore the shared embedding/output weights expected by the upstream model.
    weights = owner._restore_artifact_state_dict(weights, module)
    module.load_state_dict(weights, strict=True, assign=True)
    return module

if "runtime" in globals():
    del runtime
    gc.collect()
    torch.cuda.empty_cache()

try:
    model_code.DotsTtsCore = create_empty_core
    Model._load_artifact_module = classmethod(load_weights)
    runtime = DotsTtsRuntime.from_pretrained(
        str(checkpoint),
        precision="float16",
        optimize=False,
        max_generate_length=160,
        max_sequence_length=1024,
    )
finally:
    model_code.DotsTtsCore = original_core_constructor
    Model._load_artifact_module = original_loader_descriptor

runtime.model.requires_grad_(False)
for tensor in list(runtime.model.parameters()) + list(runtime.model.buffers()):
    assert not tensor.is_meta, "A checkpoint tensor was not loaded."
print("Model loaded. Sample rate:", runtime.sample_rate)
print("Core dtype:", next(runtime.model.core.parameters()).dtype)
print("Available CPU RAM (GiB):", round(psutil.virtual_memory().available / 2**30, 2))


2026-10-20 01:40:59.117 | INFO     | dots_tts.runtime:from_pretrained:160 - [runtime] Runtime load started: model=/root/.cache/huggingface/hub/models--dots-studio--dots.tts-base/snapshots/b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1 revision=None cache_dir=None precision=float16
2026-10-20 01:40:59.118 | DEBUG    | dots_tts.runtime:_resolve_pretrained_path:198 - [runtime] Resolving pretrained path: model=/root/.cache/huggingface/hub/models--dots-studio--dots.tts-base/snapshots/b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1 revision=None cache_dir=None
2026-10-20 01:40:59.120 | DEBUG    | dots_tts.runtime:_resolve_pretrained_path:209 - [runtime] Using local pretrained directory: path=/root/.cache/huggingface/hub/models--dots-studio--dots.tts-base/snapshots/b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1
2026-10-20 01:40:59.121 | DEBUG    | dots_tts.models.dots_tts.model:from_pretrained:465 - [model] DotsTtsModel load started: pretrained_path=/root/.cache/huggingface/hub/models--dots-studio--dots.tts-bas

Model loaded. Sample rate: 48000
Core dtype: torch.float16
Available CPU RAM (GiB): 3.74


## 2 · Reproducible prompt & baseline
Download a short reference clip from the public LibriSpeech dummy dataset, define deterministic generation settings, and run the **eager (`optimize=False`)** baseline.


In [5]:
import io
import pyarrow.parquet as pq
from huggingface_hub import HfApi, list_repo_files, hf_hub_download

DATASET = "hf-internal-testing/librispeech_asr_dummy"
DATASET_REVISION = HfApi().dataset_info(DATASET).sha
filenames = list_repo_files(DATASET, repo_type="dataset", revision=DATASET_REVISION)
parquet_names = sorted(name for name in filenames if name.endswith(".parquet"))
assert parquet_names, "The dataset has no Parquet files."
reference_file = hf_hub_download(
    DATASET, parquet_names[0], repo_type="dataset", revision=DATASET_REVISION,
)

REFERENCE_AUDIO = str(OUT / "reference.wav")
REFERENCE_TEXT = None
speech_file = pq.ParquetFile(reference_file)

for batch in speech_file.iter_batches(batch_size=1, columns=["audio", "text"]):
    row = batch.to_pylist()[0]
    audio_bytes = row["audio"]["bytes"]
    if audio_bytes is None:
        continue
    waveform, sample_rate = sf.read(io.BytesIO(audio_bytes), dtype="float32")
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    seconds = len(waveform) / sample_rate
    if 3 <= seconds <= 10:
        sf.write(REFERENCE_AUDIO, waveform, sample_rate)
        REFERENCE_TEXT = row["text"]
        break

assert REFERENCE_TEXT is not None, "No suitable clip found in this dataset file."
print(REFERENCE_TEXT)
display(Audio(REFERENCE_AUDIO))
del speech_file, batch, row, waveform, audio_bytes


clean/validation-00000-of-00001.parquet:   0%|          | 0.00/9.19M [00:00<?, ?B/s]

MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL


In [6]:
TEXT = "Hello, this is a test of the dots TTS model."
SEED = 42
STEPS = 10

def generate(steps=STEPS, profile=False):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    with torch.inference_mode():
        return runtime.generate(
            text=TEXT,
            prompt_audio_path=REFERENCE_AUDIO,
            prompt_text=REFERENCE_TEXT,
            num_steps=steps,
            ode_method="euler",
            guidance_scale=1.2,
            speaker_scale=1.5,
            profile_inference=profile,
        )


In [7]:
baseline = generate(profile=True)
baseline_audio = baseline["audio"].detach().float().cpu().squeeze().numpy()
assert np.isfinite(baseline_audio).all(), "Audio has invalid values."
sf.write(OUT / "baseline.wav", baseline_audio, baseline["sample_rate"])
baseline_profile = baseline["profiling"]
display(Audio(str(OUT / "baseline.wav")))
display(pd.DataFrame(baseline_profile).T)
del baseline


2026-10-20 01:41:52.346 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-20 01:42:29.851 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-20 01:42:29.859 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-20 01:42:29.860 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

,seconds,count,rtf
FM,8.174592,22.0,2.432914
latent_encoder,0.175685,1.0,0.052287
patch_encoder,1.059981,23.0,0.315471
LLM,1.165920,23.0,0.347000
latent_decoder,0.612628,1.0,0.182330
speaker_encoder,2.249767,1.0,0.669574
vocoder,0.000000,0.0,0.000000


## 3 · Phase 1: Inference call path
The main output of code-path tracing was locating **the repeated neural-network forward pass** within flow-matching refinement.

```text
DotsTtsRuntime.generate()
  → model.generate_audio()
  → _generate_latents_stream()
  → _decode_next_audio()
  → DiTSolver.decode_next()
  → EagerDiTRunner._decode_flow_matching()
  → ODE solver (10 NFE in the baseline)
  → _flow_matching_velocity(t, z)
       → coordinate_proj(z)
       → construct DiT input x
       → self.dit(...)  ← repeated forward pass
       → apply classifier-free guidance
  → solver updates z and repeats
  → refined latent patches → vocoder → waveform
```

**Why this matters for the GPU team:** The solver determines *when/how often* DiT is evaluated. `_flow_matching_velocity()` shows the per-evaluation inputs and expensive forward pass. The team should adapt the solver schedule, rather than arbitrarily skipping a call while retaining the old integration intervals.


In [8]:
import inspect
from dots_tts.modules.backbone.dit_inference import (
    DiTInferenceContext, DiTSolver, EagerDiTRunner, CachedDiTRunner,
    FusedAdaLNDiT, FusedQKVMultiHeadAttention,
)

def show_source(obj, max_lines=75):
    """Show a bounded, navigable excerpt instead of dumping entire classes."""
    try:
        lines, first_line = inspect.getsourcelines(obj)
        file_path = inspect.getfile(obj)
        label = getattr(obj, '__qualname__', getattr(obj, '__name__', type(obj).__name__))
        print(f'\n=== {label} ===\n{file_path}:{first_line}')
        for offset, line in enumerate(lines[:max_lines]):
            print(f'{first_line + offset:>5} | {line.rstrip()}')
        if len(lines) > max_lines:
            print(f'... {len(lines) - max_lines} more lines; use inspect.getsource(...) for full text.')
    except (TypeError, OSError) as exc:
        print(f'Source unavailable for {obj}: {exc}')

show_source(runtime.model.generate_audio, max_lines=55)
show_source(runtime.model._decode_next_audio, max_lines=60)
show_source(DiTSolver.decode_next, max_lines=90)
show_source(EagerDiTRunner._flow_matching_velocity, max_lines=100)



=== DotsTtsModel.generate_audio ===
/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py:1588
 1588 |     @torch.no_grad()
 1589 |     def generate_audio(
 1590 |         self,
 1591 |         data: dict[str, Any],
 1592 |         *,
 1593 |         precision: str,
 1594 |         ode_method: str,
 1595 |         num_steps: int,
 1596 |         guidance_scale: float,
 1597 |         speaker_scale: float = 1.5,
 1598 |     ) -> torch.Tensor:
 1599 |         latent_patches = list(
 1600 |             self._generate_latents_stream(
 1601 |                 data,
 1602 |                 precision=precision,
 1603 |                 ode_method=ode_method,
 1604 |                 num_steps=num_steps,
 1605 |                 guidance_scale=guidance_scale,
 1606 |                 speaker_scale=speaker_scale,
 1607 |             )
 1608 |         )
 1609 |         logger.debug(
 1610 |             logc("vocoder", "Vocoder decode started: latent_patch_count={}"),
 1611 |            

### Optional: dynamic call tracing
This reproduces the original model-level call tracing. It is **optional** and generates audio again. The `finally` block guarantees wrappers are removed even if inference fails. For generator functions, wrapper logs may reflect generator creation rather than iteration; the static call graph above is the source-of-truth for the full path.


In [9]:
from unittest.mock import patch
from functools import wraps

trace_names = ['_generate_latents_stream', '_decode_next_audio', '_consume_audio_patch']
patchers = []

def make_trace(name, original):
    @wraps(original)
    def traced(*args, **kwargs):
        print(f'[TRACE] call {name}', flush=True)
        return original(*args, **kwargs)
    return traced

try:
    for name in trace_names:
        original = getattr(runtime.model, name, None)
        if original is None:
            continue
        p = patch.object(runtime.model, name, make_trace(name, original))
        p.start()
        patchers.append(p)
    traced_result = generate(profile=False)
    del traced_result
    print('Call trace completed.')
finally:
    for p in reversed(patchers):
        p.stop()
    patchers.clear()
    print('Tracing hooks removed.')


2026-10-20 01:42:44.307 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-20 01:42:44.316 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-20 01:42:44.317 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-20 01:42:44.318 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

[TRACE] call _generate_latents_stream


2026-10-20 01:42:44.330 | DEBUG    | dots_tts.models.dots_tts.model:_prepare_prompt_conditioning:890 - [conditioning] Prompt speaker cache hit: key=9c87ac130eb3 prompt_samples=238080
2026-10-20 01:42:44.332 | DEBUG    | dots_tts.models.dots_tts.model:_prepare_prompt_conditioning:920 - [conditioning] Prompt latent cache hit: key=9c87ac130eb3 prompt_samples=238080
2026-10-20 01:42:44.333 | DEBUG    | dots_tts.models.dots_tts.model:_prepare_prompt_conditioning:936 - [conditioning] Prompt conditioning prepared: prompt_samples=238080 prompt_patch_count=30 speaker_scale=1.5 device=cuda:0
2026-10-20 01:42:44.335 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1429 - [decode] Latent generation prepared: schedule_audio_spans=160 prompt_patch_count=30 minimum_required_spans=31


[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch


2026-10-20 01:42:45.032 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=1


...

[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch
[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch


2026-10-20 01:42:51.449 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=20


[TRACE] call _decode_next_audio
[TRACE] call _consume_audio_patch


2026-10-20 01:42:51.843 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1499 - [decode] Latent generation completed: payload_audio_patches=21
2026-10-20 01:42:51.846 | DEBUG    | dots_tts.models.dots_tts.model:generate_audio:1609 - [vocoder] Vocoder decode started: latent_patch_count=21
2026-10-20 01:42:52.136 | DEBUG    | dots_tts.models.dots_tts.model:generate_audio:1616 - [vocoder] Vocoder decode completed: waveform_samples=161280
2026-10-20 01:42:52.137 | INFO     | dots_tts.runtime:generate:885 - [request] Generation completed: request_id=d8e2b5aeeff96d5a elapsed_seconds=7.817 audio_seconds=3.360 rtf=2.3264 sample_rate=48000


Call trace completed.
Tracing hooks removed.


## 4 · Phase 2: Tensor dependency and reuse map
A tensor is **invariant within a solve** if its value does not change between that audio patch's NFE evaluations. Invariance alone is not a new optimization: many of these inputs are already prepared once outside the eager solver loop.

| Tensor | Where it originates | Within one solve | Reuse / implication |
|---|---|---|---|
| `g_cond` | Global/speaker conditioning | Invariant | Already reused; derived CFG branches are reconstructed in eager mode |
| `input_sequence`, `cfg_input_sequence` | Previous audio-context state | Invariant | Prepared once per patch; context grows between patches |
| `attn_mask` | `_build_decode_mask()` | Invariant | Already prepared outside the eager NFE loop |
| `pos_ids` | `_build_pos_ids()` | Invariant | Already prepared outside the eager NFE loop |
| `guidance` | Guidance configuration | Invariant | Already reused |
| `g_cond_branches` | Eager `_flow_matching_velocity()` | Same intended values | Rebuilt each NFE; minor potential reuse |
| `t` | ODE solver | Changes | Determines timestep-conditioned DiT/AdaLN state |
| `z` | ODE latent state | Changes | Must update each evaluation |
| `z_proj` | `coordinate_proj(z)` | Changes | Generally recomputed when `z` changes |
| `x` | Assembled DiT input | Changes | Incorporates changing latent information |
| `pred` | `self.dit(...)` output | Changes | Recomputed velocity prediction |

**Key distinction:** These are source-level dependency findings. The *eager* loop uses the tensors above; the optimized runner processes context through its own cache-aware steps.


## 5 · Phase 2: What happens inside the DiT?
The flow-matching backbone is `FusedAdaLNDiT`, with **18 transformer blocks**. Attention uses `FusedQKVMultiHeadAttention` and a fused projection rather than independently exposed Q/K/V modules.

```text
Timestep + conditioning
   → build_condition()
   → fused_adaln()
   → run_blocks()
        → modulated hidden representation
        → fused Q/K/V projection and attention
   → DiT prediction
```

The critical dependency is **timestep → AdaLN modulation → hidden features → K/V**. The historical context may remain identical, but the resulting attention K/V can differ across refinement timesteps.


In [10]:
show_source(FusedAdaLNDiT.forward, max_lines=100)
show_source(FusedQKVMultiHeadAttention.forward, max_lines=90)

solver = runtime.model._get_dit_solver(solver_mode='flow_matching')
block = solver.context.dit.blocks[0]
print('DiT block:', type(block).__name__)
show_source(type(block).forward, max_lines=90)



=== FusedAdaLNDiT.forward ===
/content/dots.tts/src/dots_tts/modules/backbone/dit_inference.py:305
  305 |     def forward(
  306 |         self,
  307 |         x: torch.Tensor,
  308 |         timesteps: torch.Tensor,
  309 |         duration: torch.Tensor | None = None,
  310 |         attn_mask: torch.Tensor | None = None,
  311 |         pos_ids: torch.Tensor | None = None,
  312 |         g_cond: torch.Tensor | None = None,
  313 |     ) -> torch.Tensor:
  314 |         all_mods = self.fused_adaln(
  315 |             self.build_condition(timesteps, duration=duration, g_cond=g_cond)
  316 |         )
  317 |         return self.run_blocks(
  318 |             x=x, all_mods=all_mods, attn_mask=attn_mask, pos_ids=pos_ids
  319 |         )

=== FusedQKVMultiHeadAttention.forward ===
/content/dots.tts/src/dots_tts/modules/backbone/dit_inference.py:115
  115 |     def forward(
  116 |         self,
  117 |         q: torch.Tensor,
  118 |         k: torch.Tensor | None = None,
  119 

### 5.1 · Find DiT, attention and Q/K/V symbols across the repository

The DiT inspection above follows the main forward path. This search adds **file paths and line numbers** for attention-related classes and Q/K/V projection references throughout `src/dots_tts`. It helps locate tensor creation and consumption outside the functions shown above, particularly fused Q/K/V operations that would not appear under separate `q_proj`, `k_proj`, or `v_proj` names.

Treat these results as a **static source-code index**. A matching reference is not a measured runtime operation, and this cell does not test cache reuse.


In [27]:

from pathlib import Path

repo_src = Path(REPO) / "src" / "dots_tts"

print("===== DiT and attention definitions =====")

for path in sorted(repo_src.rglob("*.py")):
    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped.startswith("class ") and any(
            term in stripped.lower()
            for term in ("dit", "attention", "attn")
        ):
            print(f"{path.relative_to(repo_src)}:{i + 1}: {stripped}")

print("\n===== Attention projection references =====")

terms = (
    "q_proj",
    "k_proj",
    "v_proj",
    "to_q",
    "to_k",
    "to_v",
    "qkv",
)

for path in sorted(repo_src.rglob("*.py")):
    if "test" in path.parts:
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines):
        if any(term in line for term in terms):
            print(f"{path.relative_to(repo_src)}:{i + 1}: {line.strip()}")


===== DiT and attention definitions =====
edit_runtime.py:33: class EditRuntimeInputs(TypedDict, total=False):
edit_runtime.py:56: class DotsTtsEditRuntime(DotsTtsRuntime):
models/dots_tts/config.py:30: class _DiTConfig(ConfigBase):
models/dots_tts/edit_model.py:31: class _EditPromptFeatureCacheEntry:
models/dots_tts/edit_model.py:36: class DotsTtsEditModel(DotsTtsModel):
models/dots_tts/model.py:57: class _PromptConditioning:
modules/backbone/dit.py:60: class DiTBlock(nn.Module):
modules/backbone/dit.py:124: class DiT(nn.Module):
modules/backbone/dit_inference.py:87: class FusedQKVMultiHeadAttention(nn.Module):
modules/backbone/dit_inference.py:162: class FusedAdaLNDiT(nn.Module):
modules/backbone/dit_inference.py:341: class DiTInferenceContext:
modules/backbone/dit_inference.py:378: class DiTKvCache:
modules/backbone/dit_inference.py:455: class DiTSolverState:
modules/backbone/dit_inference.py:683: class CachedDiTRunner:
modules/backbone/dit_inference.py:955: class EagerDiTRunner:
mo

## 6 · Existing KV-cache implementation
The optimized runner (`optimize=True`) has an existing `DiTKvCache` containing historical attention `cache_k` and `cache_v`.

**Cache lifecycle**

```text
Historical audio context
  → prefill() K/V for the solver's timesteps
  → cache_k[ode_idx] / cache_v[ode_idx] (per layer)
  → NextPatchStep attention receives key_prefix / value_prefix
  → compute new_k / new_v for new/current context
  → extend historical cache for subsequent patches
```

The first patch has no historical prefix to retrieve (`FirstPatchStep`). Subsequent patches can reuse compatible historical representations via `NextPatchStep`. The optimized runner also reuses compatible rotary state, masks, and timestep-specific modulation.

**Adaptive-NFE implication:** The current cache is organized by **ODE/NFE index**, but K/V values depend on the *actual timestep*. Changing the NFE schedule may make cache entries incompatible; a future scheduler must manage cache validity and update missing historical state. This is a source-derived design constraint, not a demonstrated correctness failure.


In [11]:
# Inspect the concrete cache and runner implementations without a full source dump.
cache_module = inspect.getmodule(CachedDiTRunner)
for cls_name in ['DiTKvCache', 'FirstPatchStep', 'NextPatchStep']:
    cls = getattr(cache_module, cls_name, None)
    if cls is None:
        print(f'{cls_name}: not directly exported from {cache_module.__name__}')
        continue
    print(f'\n{cls_name}: {inspect.getfile(cls)}')
    if hasattr(cls, 'forward'):
        show_source(cls.forward, max_lines=75)

for cls, names in [
    (DiTSolver, ['_prepare_schedule']),
    (CachedDiTRunner, ['prefill', 'decode_next']),
]:
    for name in names:
        method = getattr(cls, name, None)
        if method is not None:
            show_source(method, max_lines=85)

print('\nCachedDiTRunner cache-related methods:')
for name, _ in inspect.getmembers(CachedDiTRunner, predicate=inspect.isfunction):
    if any(term in name.lower() for term in ['cache', 'prefill', 'rotary', 'mask']):
        print(' -', name)



DiTKvCache: /content/dots.tts/src/dots_tts/modules/backbone/dit_inference.py

FirstPatchStep: /content/dots.tts/src/dots_tts/modules/backbone/dit_inference.py

=== FirstPatchStep.forward ===
/content/dots.tts/src/dots_tts/modules/backbone/dit_inference.py:480
  480 |     def forward(
  481 |         self,
  482 |         z: torch.Tensor,
  483 |         *,
  484 |         current_hidden: torch.Tensor,
  485 |         all_mods: torch.Tensor,
  486 |         t: torch.Tensor | None = None,
  487 |         cfg_current_hidden: torch.Tensor | None = None,
  488 |         guidance_scale: torch.Tensor | None = None,
  489 |     ) -> torch.Tensor:
  490 |         del t
  491 |         batch_size = current_hidden.size(0)
  492 |         z_proj = self.coordinate_proj(z)
  493 |         x = concat_branches(
  494 |             [current_hidden, z_proj],
  495 |             ([cfg_current_hidden, z_proj] if cfg_current_hidden is not None else None),
  496 |         )
  497 |         vt = self.dit.ru

### 6.1 · Audit *all* existing reuse before proposing anything new

This is the part of the original investigation that must come **before** benchmarking. Separate **reuse visible in the implementation** from **reuse verified by a runtime trace**. `optimize=False` disables the optimized DiT path, but does **not** necessarily disable the autoregressive LLM KV cache or reference-clip caching.

| State / operation | Existing behavior to inspect | Reuse scope / invalidation | Research verdict |
|---|---|---|---|
| Prompt/reference features | Reference cache and speaker/prompt encoders | Same reference, preprocessing and weights; stochastic latent sampling can still differ | Existing reference reuse |
| LLM `past_key_values` | `LLMInference._run_step()` | Previous autoregressive text/audio prefix | **Already cached**, including in eager mode |
| `input_sequence`, CFG context | `EagerDiTRunner._prepare_inputs()` | Rebuilt between patches, but reused throughout each patch solve | **Already reused per patch** |
| `attn_mask`, `pos_ids` | Eager mask/position constructors | Fixed for NFE evaluations at a given patch length | **Already reused per patch** |
| Speaker/global `g_cond` | Solver inputs | Fixed within a patch | **Already reused**, CFG branch concatenation may repeat |
| `cache_k`, `cache_v` | `DiTKvCache` and `CachedDiTRunner` | Historical prefix; separated by DiT layer, branch and solver-time index | **Already cached in optimized path** |
| Rotary cosine/sine state | Cached runner positional helpers | Compatible positions, context length and mode | **Already cached where implemented** |
| Timestep schedule / AdaLN modulation | `DiTSolver._prepare_schedule*` | Compatible solver time, conditioning and NFE grid | **Already prepared / reused where compatible** |
| `z`, `z_proj`, DiT prediction | ODE solver and `_flow_matching_velocity()` | Changes each NFE evaluation | Not generally reusable |
| `g_cond_branches` | Eager `_flow_matching_velocity()` | Same values within the patch, but may be reconstructed | Possible **minor** improvement; measure before prioritizing |

**Important:** A historical *input* being unchanged does not imply its attention K/V stays unchanged across different timesteps. DiT's time-dependent AdaLN modulation changes the hidden features entering attention. The existing optimized cache is organized by compatible solver times; an adaptive-NFE scheduler cannot blindly reuse K/V by **step number** after changing the time grid.

**Benchmarking note:** Comparing `optimize=False` to `optimize=True` measures an entire optimized inference mode, **not a KV-cache-only ablation**.


In [12]:
# Inspect the EXISTING cache paths, rather than assuming cache candidates are new work.
from dots_tts.modules.backbone.llm_inference import LLMInference

print('=== LLM K/V ===')
for name in ('_run_step', 'step'):
    fn = getattr(LLMInference, name, None)
    if fn is not None:
        show_source(fn, max_lines=80)

print('\n=== DiT cache allocation and time-specific schedule ===')
for cls, names in (
    (CachedDiTRunner, ('allocate_cache', 'prefill', 'decode_next')),
    (DiTSolver, ('_prepare_schedule_flow_matching', '_decode_with_kv_cache')),
):
    for name in names:
        fn = getattr(cls, name, None)
        if fn is not None:
            show_source(fn, max_lines=110)

cache_cls = getattr(cache_module, 'DiTKvCache', None)
if cache_cls is not None:
    print('\nDiTKvCache members:', [name for name in vars(cache_cls)
          if any(key in name.lower() for key in ('cache', 'allocate', 'update', 'prefill', 'forward'))])
    show_source(cache_cls, max_lines=120)

print('\n=== Reference/prompt cache candidates ===')
for name in dir(runtime.model):
    if 'cache' in name.lower() or 'prompt' in name.lower() and 'cache' in name.lower():
        attr = getattr(runtime.model, name, None)
        print(name, type(attr).__name__)


=== LLM K/V ===

=== LLMInference._run_step ===
/content/dots.tts/src/dots_tts/modules/backbone/llm_inference.py:119
  119 |     def _run_step(
  120 |         self,
  121 |         inputs_embeds: torch.Tensor,
  122 |         past_key_values: Any | None,
  123 |         *,
  124 |         request_logits: bool,
  125 |         cache_position: torch.Tensor | None = None,
  126 |     ) -> tuple[torch.Tensor, torch.Tensor | None, Any | None]:
  127 |         outputs = self.core._llm_base_model()(
  128 |             inputs_embeds=inputs_embeds,
  129 |             past_key_values=past_key_values,
  130 |             use_cache=True,
  131 |             return_dict=True,
  132 |             cache_position=cache_position,
  133 |         )
  134 |         hidden = self.core._last_hidden_state(outputs)
  135 |         if hidden is None:
  136 |             raise RuntimeError("LLM base model did not return last_hidden_state.")
  137 |         logits = self.core._compute_lm_logits(hidden) if re

#### 6.1.1 · Compare all eager and cached runner methods

The preceding cache audit highlights the existing LLM and DiT cache entry points. This additional source inspection prints the complete available Python methods of `EagerDiTRunner` and `CachedDiTRunner`, rather than truncating selected methods. Compare where eager inference builds inputs and recomputes attention with the cached runner's **prefill, historical K/V prefix reuse and cache updates**.

This is intentionally verbose and is **source inspection only**. Reading these methods does not measure cache hits or directly validate the persisted optimized cache at runtime.


In [28]:

import inspect

for runner_cls in [EagerDiTRunner, CachedDiTRunner]:
    print(f"\n{'=' * 60}")
    print(runner_cls.__name__)
    print("Source:", inspect.getfile(runner_cls))
    print(f"{'=' * 60}")

    methods = inspect.getmembers(
        runner_cls,
        predicate=inspect.isfunction
    )

    for name, fn in methods:
        if name.startswith("__"):
            continue

        print(f"\n----- {name} -----")
        print(inspect.getsource(fn))



EagerDiTRunner
Source: /content/dots.tts/src/dots_tts/modules/backbone/dit_inference.py

----- _build_decode_mask -----
    def _build_decode_mask(
        self, *, total_len: int, fm_seq_len: int, device: torch.device
    ) -> torch.Tensor:
        mask = torch.zeros((1, total_len, total_len), dtype=torch.bool, device=device)
        latent_start = total_len - self.latent_patch_size
        block_start = int(fm_seq_len) - self.hidden_patch_size
        if block_start > 0:
            mask[:, :block_start, :block_start] = (
                torch.ones((block_start, block_start), dtype=torch.bool, device=device)
                .triu(1)
                .logical_not()
            )
        mask[:, block_start:fm_seq_len, :fm_seq_len] = True
        mask[:, block_start:fm_seq_len, latent_start:] = True
        mask[:, latent_start:, :fm_seq_len] = True
        mask[:, latent_start:, latent_start:] = True
        return mask


----- _build_pos_ids -----
    def _build_pos_ids(
        self

### 6.2 · Trace actual tensors inside the eager NFE loop

The source audit above tells us **what is designed to be cached**. The experiment below answers **what actually changes** over repeated calls. It instruments the eager flow-matching path, records input tensor metadata, selected layer K/V *projections*, time modulation and operation counts, then compares full saved tensors.

**Run order:** Complete Sections 1–6.1, including the baseline, **before** these cells. Your existing Colab instructions still apply: select **runtime version `2026.07` / Python 3.12**, and **restart the runtime after pip install**, then resume **at the next imports cell**. Do not rerun pip until a fresh runtime requires it. Preserve the T4 low-RAM model loader.

**Instrumentation limits:** Only the first **three internal decode calls** and the first/last DiT layers are traced by default, with a 128-MiB CPU snapshot budget. Hooks and patched methods are always restored in `finally`. This is not a latency benchmark, since copying tensors to CPU and synchronizing changes timing.

**Interpretation warning:** The `qkv_proj` hook sees *pre-normalization and pre-RoPE projected K/V*, not the actual persisted optimized DiT cache. Real cached K/V contents are not validated by an eager-only trace.


In [13]:
# Trace configuration (safe default for Colab T4). Run AFTER eager baseline.
assert not runtime.optimize, 'Run tensor tracing in eager mode (optimize=False).'
dit = runtime.model.core.velocity_field_predictor
TRACE_LAYERS = sorted({0, len(dit.blocks) - 1})
TRACE_CALLS = 3
SNAPSHOT_BUDGET_BYTES = 128 * 1024 * 1024  # CPU copies, not GPU cache

print('DiT:', type(dit).__name__)
print('Total blocks:', len(dit.blocks))
print('Selected blocks:', TRACE_LAYERS)
assert hasattr(dit.blocks[TRACE_LAYERS[0]].attn, 'qkv_proj'), (
    'Run baseline first to initialize the fused attention modules.'
)

def reset_tensor_trace():
    global _trace, tensor_snapshots, trace_tensor_rows, trace_decode_rows, trace_op_rows
    _trace = {'call': -1, 'eval': 0, 'time': None, 'active': False,
              'fm_len': 0, 'history_len': 0, 'saved_bytes': 0}
    tensor_snapshots = {}
    trace_tensor_rows = []
    trace_decode_rows = []
    trace_op_rows = []

reset_tensor_trace()

def capture_tensor(name, value):
    if not _trace['active'] or value is None:
        return
    if not torch.is_tensor(value):
        if not isinstance(value, (bool, int, float)):
            return
        value = torch.as_tensor(value)
    byte_count = value.numel() * value.element_size()
    index = (_trace['call'], _trace['eval'], name)
    saved = _trace['saved_bytes'] + byte_count <= SNAPSHOT_BUDGET_BYTES
    if saved:
        tensor_snapshots[index] = value.detach().cpu().clone()
        _trace['saved_bytes'] += byte_count
    trace_tensor_rows.append({
        'decode_call': _trace['call'], 'evaluation': _trace['eval'],
        'timestep': _trace['time'], 'tensor': name, 'shape': str(tuple(value.shape)),
        'dtype': str(value.dtype), 'device': str(value.device),
        'bytes': int(byte_count), 'saved_full_tensor': bool(saved),
    })


DiT: FusedAdaLNDiT
Total blocks: 18
Selected blocks: [0, 17]


In [14]:
# Instrument the exact eager runner and selected transformer operations.
# These wrappers only observe data and return the original method's output.
_original_decode = EagerDiTRunner.decode_next
_original_velocity = EagerDiTRunner._flow_matching_velocity

def _observe_decode(runner, **kwargs):
    _trace['call'] += 1
    _trace['eval'] = 0
    _trace['time'] = None
    _trace['active'] = _trace['call'] < TRACE_CALLS
    _trace['fm_len'] = int(kwargs['fm_seq_len'])
    _trace['history_len'] = max(0, _trace['fm_len'] - runner.hidden_patch_size)
    try:
        return _original_decode(runner, **kwargs)
    finally:
        trace_decode_rows.append({
            'decode_call': _trace['call'], 'sequence_length': _trace['fm_len'],
            'requested_nfe': kwargs.get('nfe'), 'solver_evaluations': _trace['eval'],
        })
        _trace['active'] = False

def _observe_velocity(runner, t, z, **kwargs):
    _trace['eval'] += 1
    if _trace['active']:
        _trace['time'] = float(t.detach().cpu()) if torch.is_tensor(t) else float(t)
        capture_tensor('t', t)
        capture_tensor('z', z)
        for name in ('input_sequence', 'cfg_sequence', 'attn_mask',
                     'pos_ids', 'g_cond', 'guidance_scale'):
            capture_tensor(name, kwargs.get(name))
    result = _original_velocity(runner, t, z, **kwargs)
    capture_tensor('velocity', result)
    return result

def _qkv_observer(layer_idx):
    def hook(module, inputs, output):
        if not _trace['active'] or not torch.is_tensor(output):
            return
        _, k, v = output.chunk(3, dim=-1)
        for label, value in (('K_projection', k), ('V_projection', v)):
            prefix = f'layer{layer_idx}.{label}'
            a = _trace['history_len']
            b = _trace['fm_len']
            capture_tensor(prefix + '.history', value[:, :a])
            capture_tensor(prefix + '.current_hidden', value[:, a:b])
            capture_tensor(prefix + '.noisy_latent', value[:, b:])
    return hook

def _operation_observer(name):
    def hook(module, inputs, output):
        if _trace['active']:
            trace_op_rows.append({'decode_call': _trace['call'],
                                  'evaluation': _trace['eval'], 'operation': name})
            if name in ('time_embedding', 'time_modulation'):
                capture_tensor(name, output)
    return hook

observed_ops = {
    'coordinate_proj(z)': runtime.model.core.coordinate_proj,
    'time_embedding': dit.time_embedder,
    'time_modulation': dit.fused_adaln,
}
for layer_idx in TRACE_LAYERS:
    observed_ops[f'layer{layer_idx}.qkv_proj'] = dit.blocks[layer_idx].attn.qkv_proj
    observed_ops[f'layer{layer_idx}.ffn'] = dit.blocks[layer_idx].ffn
print('Tracing operations:', list(observed_ops))


Tracing operations: ['coordinate_proj(z)', 'time_embedding', 'time_modulation', 'layer0.qkv_proj', 'layer0.ffn', 'layer17.qkv_proj', 'layer17.ffn']


In [15]:
# Run in eager mode. The finally block guarantees NO instrumentation leaks into benchmarking.
assert not runtime.optimize, 'If benchmarking was already run, switch back to eager first.'
reset_tensor_trace()
trace_handles = []
traced_result = None
try:
    for layer_idx in TRACE_LAYERS:
        trace_handles.append(dit.blocks[layer_idx].attn.qkv_proj.register_forward_hook(
            _qkv_observer(layer_idx)))
    for label, module in observed_ops.items():
        trace_handles.append(module.register_forward_hook(_operation_observer(label)))
    EagerDiTRunner.decode_next = _observe_decode
    EagerDiTRunner._flow_matching_velocity = _observe_velocity
    traced_result = generate(steps=STEPS, profile=False)
finally:
    EagerDiTRunner.decode_next = _original_decode
    EagerDiTRunner._flow_matching_velocity = _original_velocity
    for handle in trace_handles:
        handle.remove()
    _trace['active'] = False

assert trace_tensor_rows, 'No tensor metadata captured; check the eager inference path.'
trace_audio = traced_result['audio'].detach().float().cpu().squeeze().numpy()
sf.write(OUT / 'traced_eager.wav', trace_audio, traced_result['sample_rate'])
print('Baseline and traced WAV samples exactly equal:', np.array_equal(baseline_audio, trace_audio))
print('Snapshot budget used (MiB):', round(_trace['saved_bytes']/2**20, 2))
print('Hooks removed; benchmark can now run uninstrumented.')
del traced_result, trace_audio


2026-10-20 01:42:53.024 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-20 01:42:53.038 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-20 01:42:53.041 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-20 01:42:53.042 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

Baseline and traced WAV samples exactly equal: True
Snapshot budget used (MiB): 107.63
Hooks removed; benchmark can now run uninstrumented.


### 6.3 · Compare NFE invariants and reused historical context

The next cells compute full-tensor comparisons **only** where both snapshots were captured. A fixed tensor pointer is weaker evidence than identical full values. Within a patch we compare consecutive solver evaluations; across patches we compare only shared historical projections at **matching solver timesteps**.

For the across-patch comparison, equality is **not guaranteed** even at matching steps because the eager path recomputes the full sequence; differences can result from time conditioning, evolving context or numerical effects. A missing comparison is not a cache hit. These results demonstrate projected tensor behavior, not the optimized cache's correctness.


In [16]:
# Inventory of actual tensor sizes, calls and repeated operations.
tensors = pd.DataFrame(trace_tensor_rows)
decode_calls = pd.DataFrame(trace_decode_rows)
operations_run = pd.DataFrame(trace_op_rows)
display(decode_calls.head(12))
display(tensors[tensors['evaluation'] == 1].head(50))
if not operations_run.empty:
    display(operations_run.groupby(['decode_call', 'operation']).size()
            .rename('calls').reset_index())
print('Snapshots skipped (budget):', int((~tensors['saved_full_tensor']).sum()))

def compare_full_tensors(a, b):
    if a.shape != b.shape or a.numel() == 0:
        return None
    return {
        'exactly_equal': bool(torch.equal(a, b)),
        'max_abs_difference': float((a.float() - b.float()).abs().max().item()),
        'values_compared': int(a.numel()),
    }

within_rows = []
for (call, evaluation, name), current in tensor_snapshots.items():
    previous = tensor_snapshots.get((call, evaluation - 1, name))
    if previous is not None:
        comparison = compare_full_tensors(previous, current)
        if comparison is not None:
            within_rows.append({'decode_call': call, 'evaluation': evaluation,
                                'tensor': name, **comparison})
within = pd.DataFrame(within_rows, columns=[
    'decode_call', 'evaluation', 'tensor', 'exactly_equal',
    'max_abs_difference', 'values_compared'])
if not within.empty:
    display(within.groupby('tensor').agg(
        comparisons=('exactly_equal', 'size'),
        equal_comparisons=('exactly_equal', 'sum'),
        largest_difference=('max_abs_difference', 'max'),
    ).reset_index())
else:
    print('No within-patch pairs available. Increase trace coverage before drawing conclusions.')


,decode_call,sequence_length,requested_nfe,solver_evaluations
0,0,151,10,10
1,1,156,10,10
2,2,161,10,10
3,3,166,10,10
4,4,171,10,10
5,5,176,10,10
6,6,181,10,10
7,7,186,10,10
8,8,191,10,10
9,9,196,10,10


,decode_call,evaluation,timestep,tensor,shape,dtype,device,bytes,saved_full_tensor
0,0,1,0.0,t,(),torch.float16,cuda:0,2,True
1,0,1,0.0,z,"(1, 4, 128)",torch.float16,cuda:0,1024,True
2,0,1,0.0,input_sequence,"(1, 155, 1024)",torch.float16,cuda:0,317440,True
3,0,1,0.0,cfg_sequence,"(1, 155, 1024)",torch.float16,cuda:0,317440,True
4,0,1,0.0,attn_mask,"(1, 155, 155)",torch.bool,cuda:0,24025,True
5,0,1,0.0,pos_ids,"(1, 155)",torch.float32,cuda:0,620,True
6,0,1,0.0,g_cond,"(1, 1024)",torch.float16,cuda:0,2048,True
7,0,1,0.0,guidance_scale,(),torch.float16,cuda:0,2,True
8,0,1,0.0,time_embedding,"(2, 1024)",torch.float16,cuda:0,4096,True
9,0,1,0.0,time_modulation,"(2, 112640)",torch.float16,cuda:0,450560,True


,decode_call,operation,calls
0,0,coordinate_proj(z),10
1,0,layer0.ffn,10
2,0,layer0.qkv_proj,10
3,0,layer17.ffn,10
4,0,layer17.qkv_proj,10
5,0,time_embedding,10
6,0,time_modulation,10
7,1,coordinate_proj(z),10
8,1,layer0.ffn,10
9,1,layer0.qkv_proj,10


Snapshots skipped (budget): 0


,tensor,comparisons,equal_comparisons,largest_difference
0,attn_mask,27,27,0.000000
1,cfg_sequence,27,27,0.000000
2,g_cond,27,27,0.000000
3,guidance_scale,27,27,0.000000
4,input_sequence,27,27,0.000000
5,layer0.K_projection.current_hidden,27,0,0.511719
6,layer0.K_projection.history,27,0,1.055664
7,layer0.K_projection.noisy_latent,27,0,1.265625
8,layer0.V_projection.current_hidden,27,0,0.710449
9,layer0.V_projection.history,27,0,0.772217


In [17]:
# Old-history comparisons between consecutive decode calls at the same t.
across_rows = []
for (call, evaluation, name), current in tensor_snapshots.items():
    if call == 0 or not name.endswith('.history'):
        continue
    previous = tensor_snapshots.get((call - 1, evaluation, name))
    previous_t = tensor_snapshots.get((call - 1, evaluation, 't'))
    current_t = tensor_snapshots.get((call, evaluation, 't'))
    if previous is None or previous_t is None or current_t is None:
        continue
    if not torch.equal(previous_t, current_t):
        continue  # NFE index does NOT imply matching real solver time
    if current.shape[1] < previous.shape[1]:
        continue
    old_prefix = current[:, :previous.shape[1]]
    comparison = compare_full_tensors(previous, old_prefix)
    if comparison is not None:
        across_rows.append({'previous_call': call-1, 'decode_call': call,
                            'evaluation': evaluation, 'tensor': name, **comparison})
across = pd.DataFrame(across_rows, columns=[
    'previous_call', 'decode_call', 'evaluation', 'tensor',
    'exactly_equal', 'max_abs_difference', 'values_compared'])
if not across.empty:
    display(across.groupby('tensor').agg(
        comparisons=('exactly_equal', 'size'),
        equal_comparisons=('exactly_equal', 'sum'),
        largest_difference=('max_abs_difference', 'max'),
    ).reset_index())
else:
    print('No matching history tensors captured. Do not claim cross-patch equality.')


,tensor,comparisons,equal_comparisons,largest_difference
0,layer0.K_projection.history,20,10,0.003906
1,layer0.V_projection.history,20,10,0.003906
2,layer17.K_projection.history,20,10,0.093750
3,layer17.V_projection.history,20,10,0.058594


### 6.4 · Reuse decisions before the benchmark

After inspecting the summaries, document the result for each category:

- **Already reused by eager:** patch context (`input_sequence` / `cfg_sequence`), mask, position IDs, global conditioning.
- **Already cached by optimized source:** separate historical prefix `cache_k`/`cache_v` for compatible solver evaluations/layers; cache lifecycle includes prefill, retrieving a historical prefix and updating for later patches.
- **Time dependent:** `t`, AdaLN modulation and generally K/V projected from modulated hidden representations; do not reuse across changed timesteps without verification.
- **Latent dependent:** `z`, `coordinate_proj(z)`, `velocity` and the current-patch DiT prediction; these need per-evaluation work.
- **Potential additional minor reuse:** CFG branch tensor construction when the branch values are unchanged; benefit requires benchmarking.

**For the GPU team:** Cache validity should be based on actual solver-time identity and the conditioning/context that produced the state, rather than assuming two different NFE schedules share a cache entry at index `i`. The most important practical question is how much DiT work an adaptive scheduler *actually* avoids once cache invalidation/rebuilding costs are included.

**Observed in the executed eager trace (10 NFE, first/last DiT layers):** The within-patch summary recorded 27/27 exact comparisons for `input_sequence`, `cfg_sequence`, `attn_mask`, `pos_ids` and `g_cond`; `t`, `z`, time embedding/modulation and the sampled K/V projections were not identical between consecutive evaluations. No snapshots were skipped by the budget. These results are limited to the traced internal decode calls and selected layers.

For the cross-patch experiment, the same history region was compared between consecutive internal decode calls at **matching recorded solver times**:

| Historical projection | Exact matches | Largest absolute difference |
|---|---:|---:|
| Layer 0, K | 10 of 20 | 0.003906 |
| Layer 0, V | 10 of 20 | 0.003906 |
| Layer 17, K | 10 of 20 | 0.093750 |
| Layer 17, V | 10 of 20 | 0.058594 |

**Interpretation:** Half of the sampled historical *projections* matched exactly, while half did not, even under the matching-time filter. Do not equate this with a 50% cache-hit rate or with correctness of `DiTKvCache`. The eager trace captured **pre-normalization/pre-RoPE projections**, not the persisted K/V of the optimized path. Any proposed cross-patch reuse still needs its own validity and equivalence checks.


In [18]:
# Export the evidence collected before the optimized-mode benchmark.
for filename, frame in (
    ('tensor_inventory.csv', tensors),
    ('decode_calls.csv', decode_calls),
    ('operations.csv', operations_run),
    ('within_patch.csv', within),
    ('across_patches.csv', across),
):
    frame.to_csv(OUT / filename, index=False)

print('Source-derived cache map and traced tensor comparisons ready.')
print('Saved tensor evidence in', OUT)


Source-derived cache map and traced tensor comparisons ready.
Saved tensor evidence in /content/rababb_kv_map


## 7 · Eager vs. optimized inference benchmark
Same prompt, reference clip, seed, Euler solver, guidance, hardware and **10 NFE**. Each mode gets a warm-up (not measured), followed by two timed trials. Timing uses GPU synchronization. Peak allocated GPU memory is tracked.

**Important:** `optimize=True` changes more than KV caching (including compiled/optimized paths), so this is an **optimized-mode benchmark**, not a KV-cache-only ablation. The sound of the outputs should also be checked; equal duration alone does not demonstrate equal quality.

The benchmark saves four WAV files and `eager_vs_cached_results.csv` under `/content/rababb_kv_map/`.


In [19]:
import time
import gc
import pandas as pd
import torch
import soundfile as sf
from IPython.display import Audio, display

assert "runtime" in globals(), "Run your model-loading cell first."
assert "generate" in globals(), "Run your existing generate() function cell first."

def set_inference_mode(cached):
    runtime.optimize = cached
    runtime.model.set_optimize(
        cached,
        max_sequence_length=runtime.max_sequence_length,
    )
    print("Mode:", "CACHED" if cached else "EAGER")


def benchmark_run(mode, trial):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    result = generate(steps=STEPS, profile=True)

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

    profiling = result.get("profiling") or {}
    fm = profiling.get("FM") or {}

    audio = result["audio"].detach().float().cpu().squeeze().numpy()

    output_path = OUT / f"{mode}_trial_{trial}.wav"
    sf.write(output_path, audio, result["sample_rate"])

    row = {
        "mode": mode,
        "trial": trial,
        "total_time_s": elapsed,
        "fm_time_s": fm.get("seconds"),
        "fm_calls": fm.get("count"),
        "audio_duration_s": len(audio) / result["sample_rate"],
        "rtf": elapsed / (len(audio) / result["sample_rate"]),
        "peak_gpu_gb": peak_memory,
    }

    del result, audio
    gc.collect()

    print(row)
    return row

print("Benchmark functions ready!")


Benchmark functions ready!


In [20]:
set_inference_mode(False)

# Warm-up run: excluded from measurements
print("Warming up eager inference...")
warmup = generate(steps=STEPS, profile=False)
del warmup

torch.cuda.synchronize()

# Two measured runs
eager_results = [
    benchmark_run("eager", 1),
    benchmark_run("eager", 2),
]

display(pd.DataFrame(eager_results))


2026-10-20 01:43:01.249 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-20 01:43:01.257 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-20 01:43:01.259 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-20 01:43:01.261 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

Mode: EAGER
Warming up eager inference...


2026-10-20 01:43:01.952 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=1
2026-10-20 01:43:05.306 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=10
2026-10-20 01:43:08.683 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=20
2026-10-20 01:43:08.992 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1499 - [decode] Latent generation completed: payload_audio_patches=21
2026-10-20 01:43:08.994 | DEBUG    | dots_tts.models.dots_tts.model:generate_audio:1609 - [vocoder] Vocoder decode started: latent_patch_count=21
2026-10-20 01:43:09.285 | DEBUG    | dots_tts.models.dots_tts.model:generate_audio:1616 - [vocoder] Vocoder decode completed: waveform_samples=161280
2026-10-20 01:43:09.287 | INFO     | dots_tts.runtime:ge

{'mode': 'eager', 'trial': 1, 'total_time_s': 11.521701572999973, 'fm_time_s': 8.755951500000151, 'fm_calls': 22, 'audio_duration_s': 3.36, 'rtf': 3.4290778491071348, 'peak_gpu_gb': 5.235479354858398}


2026-10-20 01:43:22.175 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=1
2026-10-20 01:43:24.916 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=10
2026-10-20 01:43:27.990 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=20
2026-10-20 01:43:28.295 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1499 - [decode] Latent generation completed: payload_audio_patches=21
2026-10-20 01:43:28.297 | DEBUG    | dots_tts.models.dots_tts.model:generate_audio:1609 - [vocoder] Vocoder decode started: latent_patch_count=21
2026-10-20 01:43:28.802 | DEBUG    | dots_tts.models.dots_tts.model:generate_audio:1616 - [vocoder] Vocoder decode completed: waveform_samples=161280
2026-10-20 01:43:28.803 | INFO     | dots_tts.utils.prof

{'mode': 'eager', 'trial': 2, 'total_time_s': 7.345341914999949, 'fm_time_s': 5.443122062000043, 'fm_calls': 22, 'audio_duration_s': 3.36, 'rtf': 2.1861136651785564, 'peak_gpu_gb': 5.235479354858398}


,mode,trial,total_time_s,fm_time_s,fm_calls,audio_duration_s,rtf,peak_gpu_gb
0,eager,1,11.521702,8.755952,22,3.36,3.429078,5.235479
1,eager,2,7.345342,5.443122,22,3.36,2.186114,5.235479


In [21]:
cached_results = []
cached_error = None

try:
    set_inference_mode(True)

    print("Warming up cached inference...")
    warmup = generate(steps=STEPS, profile=False)
    del warmup

    torch.cuda.synchronize()

    cached_results = [
        benchmark_run("cached", 1),
        benchmark_run("cached", 2),
    ]

    display(pd.DataFrame(cached_results))

    solver = runtime.model._get_dit_solver(
        solver_mode="flow_matching"
    )

    print("Solver optimize enabled:", solver.optimize)

except Exception as exc:
    cached_error = repr(exc)
    print("Cached inference failed:")
    print(cached_error)

    import traceback
    traceback.print_exc(limit=8)


2026-10-20 01:43:29.294 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-20 01:43:29.303 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-20 01:43:29.305 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-20 01:43:29.306 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

Mode: CACHED
Warming up cached inference...


W1020 01:44:02.211000 1144 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
2026-10-20 01:45:40.874 | DEBUG    | dots_tts.modules.backbone.llm_inference:_get_static_token_step:167 - [compile] Compiled LLM static decode step: signature=(1024, 1536, torch.float16, 'cuda:0')
2026-10-20 01:46:16.372 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=1
2026-10-20 01:46:17.092 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=10
2026-10-20 01:46:17.878 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=20
2026-10-20 01:46:17.958 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1499 - [decode] Latent generation completed: payload_audio_patches=21
2026-10-20 01:46:17.959 | DEBUG    | dots_tts.

{'mode': 'cached', 'trial': 1, 'total_time_s': 2.793731579999985, 'fm_time_s': 1.6610514720000538, 'fm_calls': 22, 'audio_duration_s': 3.36, 'rtf': 0.8314677321428527, 'peak_gpu_gb': 5.586620807647705}


2026-10-20 01:46:23.278 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=1
2026-10-20 01:46:23.972 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=10
2026-10-20 01:46:24.755 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1478 - [decode] Latent generation progress: payload_audio_patches=20
2026-10-20 01:46:24.835 | DEBUG    | dots_tts.models.dots_tts.model:_generate_latents_stream:1499 - [decode] Latent generation completed: payload_audio_patches=21
2026-10-20 01:46:24.836 | DEBUG    | dots_tts.models.dots_tts.model:generate_audio:1609 - [vocoder] Vocoder decode started: latent_patch_count=21
2026-10-20 01:46:25.338 | DEBUG    | dots_tts.models.dots_tts.model:generate_audio:1616 - [vocoder] Vocoder decode completed: waveform_samples=161280
2026-10-20 01:46:25.339 | INFO     | dots_tts.utils.prof

{'mode': 'cached', 'trial': 2, 'total_time_s': 2.8128625670001384, 'fm_time_s': 1.6663699650005128, 'fm_calls': 22, 'audio_duration_s': 3.36, 'rtf': 0.8371614782738508, 'peak_gpu_gb': 5.586620807647705}


,mode,trial,total_time_s,fm_time_s,fm_calls,audio_duration_s,rtf,peak_gpu_gb
0,cached,1,2.793732,1.661051,22,3.36,0.831468,5.586621
1,cached,2,2.812863,1.666370,22,3.36,0.837161,5.586621


Solver optimize enabled: True


In [22]:
results = pd.DataFrame(eager_results + cached_results)

display(results)

summary = results.groupby("mode")[
    [
        "total_time_s",
        "fm_time_s",
        "fm_calls",
        "audio_duration_s",
        "rtf",
        "peak_gpu_gb",
    ]
].median(numeric_only=True)

print("===== MEDIAN COMPARISON =====")
display(summary)

if "cached" in summary.index and "eager" in summary.index:
    eager_time = summary.loc["eager", "total_time_s"]
    cached_time = summary.loc["cached", "total_time_s"]

    speedup = eager_time / cached_time
    reduction = (1 - cached_time / eager_time) * 100

    print(f"Overall speedup: {speedup:.2f}x")
    print(f"Time reduction: {reduction:.1f}%")

    eager_fm = summary.loc["eager", "fm_time_s"]
    cached_fm = summary.loc["cached", "fm_time_s"]

    if pd.notna(eager_fm) and pd.notna(cached_fm) and cached_fm > 0:
        print(f"Flow-matching speedup: {eager_fm / cached_fm:.2f}x")

    print("\n===== EAGER AUDIO =====")
    display(Audio(str(OUT / "eager_trial_1.wav")))

    print("===== CACHED AUDIO =====")
    display(Audio(str(OUT / "cached_trial_1.wav")))

results.to_csv(OUT / "eager_vs_cached_results.csv", index=False)

print("Saved results to:", OUT / "eager_vs_cached_results.csv")


,mode,trial,total_time_s,fm_time_s,fm_calls,audio_duration_s,rtf,peak_gpu_gb
0,eager,1,11.521702,8.755952,22,3.36,3.429078,5.235479
1,eager,2,7.345342,5.443122,22,3.36,2.186114,5.235479
2,cached,1,2.793732,1.661051,22,3.36,0.831468,5.586621
3,cached,2,2.812863,1.666370,22,3.36,0.837161,5.586621


===== MEDIAN COMPARISON =====


,total_time_s,fm_time_s,fm_calls,audio_duration_s,rtf,peak_gpu_gb
mode,,,,,,
cached,2.803297,1.663711,22.0,3.36,0.834315,5.586621
eager,9.433522,7.099537,22.0,3.36,2.807596,5.235479


Overall speedup: 3.37x
Time reduction: 70.3%
Flow-matching speedup: 4.27x

===== EAGER AUDIO =====


===== CACHED AUDIO =====


Saved results to: /content/rababb_kv_map/eager_vs_cached_results.csv


### Previously recorded T4 results (original notebook)
**These values were measured in the uploaded source notebook; they are not from a fresh execution of this cleaned copy.** Medians of two timed trials per mode:

| Metric | Eager | Optimized (`optimize=True`) |
|---|---:|---:|
| Total generation | **8.06 s** | **2.97 s** |
| Flow-matching time | 6.07 s | 1.78 s |
| Flow-matching calls | 22 | 22 |
| Audio duration | 3.36 s | 3.36 s |
| Real-time factor | 2.40 | 0.88 |
| Peak allocated GPU memory | 5.24 GiB | 5.59 GiB |

**Recorded speedups:** **2.72× end-to-end**, **3.41× flow matching**, using the same fixed NFE. This does *not* prove that lowering NFE preserves audio quality, nor isolate KV caching's contribution.


### 7.1 · Interpreting the measured eager vs. optimized benchmark

**Current executed notebook, NVIDIA T4, FP16, Euler and 10 NFE:** One warm-up per mode was excluded, and two timed trials per mode produced the following medians:

| Metric | Eager (`optimize=False`) | Optimized (`optimize=True`) |
|---|---:|---:|
| Total generation | **9.43 s** | **2.80 s** |
| Flow-matching time | 7.10 s | 1.66 s |
| Flow-matching calls | 22 | 22 |
| Audio duration | 3.36 s | 3.36 s |
| Real-time factor | 2.81 | 0.83 |
| Peak allocated GPU memory | 5.24 GiB | 5.59 GiB |

This corresponds to a **3.37× end-to-end speedup** and **4.27× flow-matching speedup** for the entire optimized mode. These are distinct from the **earlier** 8.06 s eager / 2.97 s optimized medians recorded above. The current eager timings were 11.52 s and 7.35 s, a wide two-trial spread, so the speedup is **preliminary**, not a stable long-run estimate.

The optimized/eager difference is an **end-to-end `optimize=True` vs `optimize=False` comparison**, *not* an isolated attention KV-cache ablation. Compilation, fused kernels, memory layout and other optimized paths can contribute. Listen to the WAVs and use ASR/speaker-similarity evaluation for defensible quality comparisons; matching audio duration is insufficient.

The trace cells in Section 6 were intentionally not timed for benchmarking. Timing can vary with runtime, warm-up, prompt-cache state and compilation. The cache also depends on actual solver timesteps: **step index `i` under different NFE schedules is not automatically a compatible K/V cache key**.


In [23]:
# Write reproducibility metadata and an evidence-based handoff without invented measurements.
import hashlib
configuration = {
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'source_revision': SOURCE_REVISION,
    'precision': 'float16',
    'hardware': torch.cuda.get_device_name(0),
    'colab_runtime_target': '2026.07',
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'seed': SEED,
    'text': TEXT,
    'reference_text': REFERENCE_TEXT,
    'reference_sha256': hashlib.sha256(Path(REFERENCE_AUDIO).read_bytes()).hexdigest(),
    'solver': 'euler',
    'nfe': STEPS,
    'guidance_scale': 1.2,
    'speaker_scale': 1.5,
    'traced_mode': 'eager (optimize=False)',
    'trace_layers': TRACE_LAYERS,
    'trace_internal_decode_calls': TRACE_CALLS,
    'snapshot_budget_bytes': SNAPSHOT_BUDGET_BYTES,
    'benchmark_modes': ['eager', 'optimized'],
    'benchmark_warmups_per_mode': 1,
    'benchmark_trials_per_mode': 2,
    'benchmark_note': 'Full optimized-mode comparison; not an isolated KV-cache ablation.',
}
(OUT / 'configuration.json').write_text(json.dumps(configuration, indent=2))
if 'baseline_profile' in globals():
    (OUT / 'stage_profile.json').write_text(json.dumps(baseline_profile, indent=2))

lines = [
    '# dots.tts existing KV/cache analysis and GPU handoff', '',
    '## Source-derived existing reuse',
    '- LLM autoregressive past_key_values are used independently of optimized DiT.',
    '- Eager inputs, masks, positions and global conditioning are reused within each patch solve.',
    '- Cached DiT reuses compatible historical prefix K/V per layer/solver evaluation and prepares time-dependent state.',
    '- The newest/current hidden/latent work must be recomputed as required by the cached implementation.',
    '- Changing NFE can invalidate the cache when the actual solver times differ.', '',
    '## Runtime evidence',
    f'- Captured {len(tensors)} tensor metadata records over {len(decode_calls)} internal decode calls.',
    f'- Within-patch comparisons: {len(within)}; matching-timestep history comparisons: {len(across)}.',
    f'- Snapshots skipped due to CPU budget: {int((~tensors["saved_full_tensor"]).sum())}.',
    '- Check tensor_inventory.csv, operations.csv, within_patch.csv and across_patches.csv.',
    '- K/V projections are pre-normalization/pre-RoPE; optimized cache contents were not directly traced.', '',
    '## Eager vs optimized benchmark',
    '- This compares complete optimize=False vs optimize=True inference modes.',
]
if 'summary' in globals() and {'eager', 'cached'}.issubset(summary.index):
    eager_s = float(summary.loc['eager', 'total_time_s'])
    cached_s = float(summary.loc['cached', 'total_time_s'])
    if cached_s > 0:
        lines.append(f'- Median eager {eager_s:.3f}s, optimized {cached_s:.3f}s; end-to-end {eager_s/cached_s:.2f}x.')
else:
    lines.append('- A paired measured comparison is unavailable in this run.')
lines += ['', '## Follow-up',
          '- Confirm correctness and quality for changing NFE before applying cache state to an adaptive scheduler.',
          '- Measure cache rebuild/invalidations and actual GPU kernel costs, not only Python operation counts.',
          '- See configuration.json for reproducibility settings.']
(OUT / 'gpu_handoff.md').write_text('\n'.join(lines) + '\n')
print('Saved configuration.json and gpu_handoff.md')


Saved configuration.json and gpu_handoff.md


In [26]:
# Optional: download everything for the GPU team. Existing Colab run notes still apply.
# This is intentionally separate from inference to avoid restarting or installing packages.
from google.colab import files
import shutil
archive_path = shutil.make_archive('/content/rababb_dots_tts_inference_codepath_tensor_analysis', 'zip', OUT)
print('Bundle:', archive_path)
files.download(archive_path)


Bundle: /content/rababb_dots_tts_inference_codepath_tensor_analysis.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8 · GPU-team handoff
**What this research established**

1. **Where to intervene:** `DiTSolver` sets up the refinement path; `_flow_matching_velocity()` executes the repeated DiT forward pass.
2. **What is constant:** Speaker/global conditioning, audio context, masks, and positions remain invariant *within a patch*; most are already reused.
3. **What must be recomputed:** The ODE latent `z`, timestep-dependent states, and DiT predictions evolve across refinement steps.
4. **What is already cached:** Historical K/V per compatible NFE index/layer, plus some positional and AdaLN state.
5. **What adaptive scheduling must address:** Cached K/V is timestep-conditioned; changing NFE requires matching or rebuilding compatible history rather than assuming index-based reuse is always valid.
6. **What to compare against:** The executed 10-NFE benchmark measured **9.43 s eager vs. 2.80 s optimized**, or **3.37× end-to-end speedup**; the **earlier run** measured 8.06 s vs. 2.97 s (2.72×). Both compare whole modes, not KV caching in isolation.

**Additional measured tensor evidence:** The current eager trace found 27/27 exact within-patch comparisons for conditioning, model context, positions and masks. For each of the four sampled historical K/V projections, 10 of 20 matching-time cross-patch comparisons were exactly equal. This is **not** the optimized cache hit rate; those hooks capture projections before normalization and rotary transformations. See the Section 6.4 table and exported CSVs.

**Earlier profiling finding (separate experiment):** DiT forward calls accounted for approximately **98.5% of traced repeated-operation time**. This figure comes from the earlier profiling analysis, **not** the operation-count hooks or the fresh eager/optimized timing benchmark in this notebook. It motivates further DiT investigation but does not quantify an isolated KV-cache benefit.

**Benchmark limitation:** The current eager trials varied between 11.52 s and 7.35 s; with only two measured trials per mode, treat 3.37× as preliminary. The latest optimized benchmark also had greater peak allocated GPU memory (5.59 GiB vs 5.24 GiB).

**Not shown by this research:** A new major unimplemented caching optimization, an adaptive step-selection policy, or a quality-preserving reduction of NFE. Those require further experiments (e.g., NFE sweeps plus speech-quality evaluation).

**Original notebook:** `WATai_dots_tts_metrics_Rababb.ipynb` retains the full exploratory history and raw execution outputs. This cleaned copy is the presentation/reproduction version.
